# DistilBERT deep-learning stage (run in Google Colab)

This fine-tunes a **DistilBERT** transformer on your reviewed emails, evaluates it on a
held-out test set, and tests the same AI-style phishing examples from your PC demo
(including the courier email your baseline got wrong).

**Do these once first:**
1. Menu **Runtime > Change runtime type > Hardware accelerator = T4 GPU > Save**.
2. Upload your converted dataset `reviewed_mail.csv` to **Google Drive** (drag it into
   [drive.google.com](https://drive.google.com) root, or any folder you remember).

Then run each cell top to bottom (click a cell, press **Shift+Enter**). Approve the Google
Drive permission pop-up. Nothing leaves Google; training runs on Colab's free GPU.


### 1. Install libraries and confirm the GPU is on


In [ ]:
!pip -q install "transformers>=4.46,<5.0" "datasets>=2.19" "accelerate>=0.30" scikit-learn
import torch
print("CUDA GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print(">> TURN ON THE GPU: Runtime > Change runtime type > T4 GPU, then re-run this cell.")
else:
    print("GPU:", torch.cuda.get_device_name(0))


### 2. Get the project code and connect Google Drive


In [ ]:
import os
!test -d proj || git clone -b arena/01a038ce-adaptive-email-spam-detection https://github.com/leegongyi2004-art/adaptive-email-spam-detection.git proj
%cd /content/proj

from google.colab import drive
drive.mount("/content/drive")
print("\nFiles in your Drive root:")
!ls "/content/drive/MyDrive" | head -30


### 3. Point at your dataset and make a balanced subset

If you put `reviewed_mail.csv` somewhere other than the Drive root, change the path in
`CSV_IN` below (e.g. `/content/drive/MyDrive/Colab Notebooks/reviewed_mail.csv`).


In [ ]:
import pandas as pd, os

CSV_IN = "/content/drive/MyDrive/reviewed_mail.csv"   # <-- change if needed
assert os.path.exists(CSV_IN), f"Not found: {CSV_IN} - check the path above."

df = pd.read_csv(CSV_IN)
df["label"] = df["label"].astype(int)
df = df.dropna(subset=["raw_email"])
print("Loaded", len(df), "emails  | class counts:", df.label.value_counts().to_dict())

PER_CLASS = 12000   # emails per class for training+validation (keeps free-GPU time ~20-30 min)
TEST_PER  = 500     # emails per class held out ONLY for scoring

test = df.groupby("label").sample(n=TEST_PER, random_state=1)
rest = df.drop(test.index)
trainval = rest.groupby("label").sample(n=PER_CLASS, random_state=42)

os.makedirs("data", exist_ok=True)
trainval.to_csv("data/colab_trainval.csv", index=False)
test.to_csv("data/colab_test.csv", index=False)
print(f"Train+val: {len(trainval)}   Test: {len(test)}")


### 4. Fine-tune DistilBERT on the GPU (~20-30 minutes)

It prints validation loss each epoch and saves the model. Keep the tab open; Colab can
disconnect if left idle too long.


In [ ]:
!python -m spam_detection.train_transformer data/colab_trainval.csv --output models/distilbert-email --max-length 256 --epochs 3 --batch-size 16


### 5. Evaluate DistilBERT on the held-out test set (screenshot for your report)


In [ ]:
import pandas as pd, time
from sklearn.metrics import classification_report, roc_auc_score
from spam_detection.transformer import TransformerEmailDetector

det = TransformerEmailDetector("models/distilbert-email", max_length=256, threshold=0.5)
test = pd.read_csv("data/colab_test.csv")

t0 = time.time()
probs = [det.predict(e).spam_probability for e in test["raw_email"]]
secs = time.time() - t0
preds = [1 if p >= 0.5 else 0 for p in probs]
y = test["label"].tolist()

print(classification_report(y, preds, target_names=["ham (legit)", "spam/phish"], digits=3))
print("ROC-AUC:", round(roc_auc_score(y, probs), 4))
print(f"latency: {secs/len(y)*1000:.0f} ms/email on GPU")
print("\nCompare to your TF-IDF baseline: accuracy 0.992, ROC-AUC 1.000, ~16 ms/email on CPU.")


### 6. The AI-style phishing test - including the courier email your baseline missed

This runs the same 12 examples from `test_ai_phishing.py` through the transformer. Watch
the legitimate **"Your delivery is out for delivery"** email especially - the baseline
flagged it; the transformer understands "no payment required" and should pass it.


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("aitest", "examples/test_ai_phishing.py")
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

def show(emails, expected, title):
    print(f"\n=== {title} ===")
    ok = 0
    for raw in emails:
        p = det.predict(raw)
        good = (p.label == expected); ok += good
        subject = next((l for l in raw.splitlines() if l.startswith("Subject:")), "")[9:]
        print(f"  [{'OK' if good else 'XX'}] {p.label:5s} p={p.spam_probability:6.2%}  {subject}")
    return ok, len(emails)

a, at = show(mod.AI_PHISHING, "spam", "AI-style phishing (should be SPAM)")
b, bt = show(mod.LEGIT,      "ham",  "Legitimate mail (should be HAM)")
print(f"\nPhishing caught: {a}/{at}   Legit passed: {b}/{bt}")
print("(Synthetic examples - report as an illustrative demonstration, not proof of AI-authorship detection.)")


### 7. Save the trained model to your Google Drive


In [ ]:
!mkdir -p "/content/drive/MyDrive/spam_model"
!cp -r models/distilbert-email "/content/drive/MyDrive/spam_model/"
print("Saved to Google Drive: MyDrive/spam_model/distilbert-email")
